# Load the Data Set 

In [25]:

import numpy as np
import pandas as pd

In [26]:
df = pd.read_csv("powerplant_data.csv")

In [27]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


# AT => Temperature
# v => Vaccum
# AP => Pressure 
# RH => Humidity

# PE => Produced Energy 

In [28]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [29]:
x = df.drop("PE" , axis = 1)
y = df ["PE"]


In [30]:
y.head()

0    480.48
1    445.75
2    438.76
3    453.09
4    464.43
Name: PE, dtype: float64

In [31]:
# Split out data
 
#import sklearn
from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2 , random_state=42)


In [32]:
x_train

,AT,V,AP,RH
5487,25.24,63.47,1011.30,66.21
3522,26.09,70.40,1007.41,85.37
6916,26.63,73.68,1015.15,85.13
7544,32.06,71.85,1007.90,56.44
7600,28.70,71.64,1007.11,69.85
...,...,...,...,...
5734,26.25,61.02,1011.47,71.22
5191,29.17,64.79,1016.43,61.05
5390,18.00,43.70,1015.40,61.28
860,26.73,68.84,1010.75,66.83


In [33]:
x_test

,AT,V,AP,RH
2513,29.70,57.35,1005.63,57.35
9411,25.71,71.64,1008.85,77.31
8745,17.83,44.92,1025.04,70.58
9085,9.46,41.40,1026.78,87.58
4950,29.90,64.79,1016.90,48.24
...,...,...,...,...
7204,20.46,51.43,1010.06,83.79
1599,29.70,67.17,1007.31,66.56
5697,14.64,39.58,1011.46,71.90
350,29.47,71.32,1008.07,67.00


In [34]:
df.shape

(9568, 5)

In [35]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)

x_test_scaled = scaler.transform(x_test)

In [36]:
x_train_scaled

array([[ 0.74805289,  0.72006931, -0.32660017, -0.49711722],
       [ 0.86181948,  1.26515721, -0.98521113,  0.8181501 ],
       [ 0.93409473,  1.52314975,  0.32523844,  0.80167494],
       ...,
       [-0.22097078, -0.834965  ,  0.36756563, -0.83554456],
       [ 0.94747903,  1.14245344, -0.41971997, -0.45455637],
       [-1.77355014, -1.19049131,  1.92520594,  0.91837402]],
      shape=(7654, 4))

In [37]:
x_test_scaled


array([[ 1.34499288,  0.23869298, -1.28658067, -1.10532538],
       [ 0.81095912,  1.36269098, -0.74140656,  0.26485915],
       [-0.2437241 , -0.73900436,  1.99970178, -0.19713193],
       ...,
       [-0.67068342, -1.15902881, -0.29951077, -0.10651852],
       [ 1.31420898,  1.33752097, -0.87346737, -0.44288647],
       [-0.2611237 , -0.27021304,  0.37433797,  1.10646548]],
      shape=(1914, 4))

In [38]:
import torch
import torch.nn as nn

x_train_tensor = torch.tensor(x_train_scaled , dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

x_test_tensor = torch.tensor(x_test_scaled , dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)


In [39]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

In [40]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

### Deep Learning

In [41]:
# Define our ANN Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(x_train.shape[1], 6),
            nn.ReLU(),
    
            # 2nd hidden layer
            nn.Linear(6, 6),
            nn.ReLU(),
    
            # output layer
            nn.Linear(6, 1),
        )

    def forward(self, x):
        return self.model(x)

In [42]:
import torch.optim as optim

model = ANN()

# loss, optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [43]:
# Train the ANN
train_losses = []
val_losses = []

best_val_loss = float("inf")

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # tot training loss for 1 epoch
    
    for xb, yb in train_loader:
        # xb = features of 1 batch
        # yb = labels of 1 batch
        optimizer.zero_grad()
        
        outputs = model(xb) # forward prop....predicted outputs for this batch
        loss = crietrion(outputs, yb) # compute loss
        loss.backward() # back prop.. compute gradients
        optimizer.step() # params update
        
        running_loss += loss.item() # loss is a tensor => py float

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): # no gradients compute
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt") #.pt or .pth

epoch 1/100 ==> train loss = 205104.3888671875 & val loss = 200592.734375
epoch 2/100 ==> train loss = 185491.22779947918 & val loss = 162694.59375
epoch 3/100 ==> train loss = 128061.567578125 & val loss = 91456.1640625
epoch 4/100 ==> train loss = 61148.50642089844 & val loss = 38320.40234375
epoch 5/100 ==> train loss = 28881.62663574219 & val loss = 23252.439453125
epoch 6/100 ==> train loss = 21014.169576009113 & val loss = 18756.8203125
epoch 7/100 ==> train loss = 17177.98829752604 & val loss = 15146.9013671875
epoch 8/100 ==> train loss = 13626.273295084635 & val loss = 11655.3388671875
epoch 9/100 ==> train loss = 10206.48814086914 & val loss = 8418.2705078125
epoch 10/100 ==> train loss = 7141.747474161783 & val loss = 5640.9130859375
epoch 11/100 ==> train loss = 4652.009627278646 & val loss = 3531.453857421875
epoch 12/100 ==> train loss = 2887.221973164876 & val loss = 2150.699462890625
epoch 13/100 ==> train loss = 1759.6728337605794 & val loss = 1317.03173828125
epoch 14

In [44]:
import matplotlib.pyplot as plt 

loss_df = pd.DataFrame({
     "Training Loss": train_losses,
     "Validation Loss" :val_losses

})

plt.plot(loss_df["Training Loss "] , label = " Training Loss")
plt.plot(loss_df[" Validation Loss"] , label = " Validation Loss")

plt.xlabel

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import matplotlib.pyplot as plt

loss_df = pd.DataFrame({
    "Training Loss": train_losses,
    "Validation Loss": val_losses
})

plt.plot(loss_df["Training Loss"], label = "Training Loss")
plt.plot(loss_df["Validation Loss"], label = "Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Loading the best model
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [ ]:
import torch.nn as nn

class ANNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(4, 6),   # matches [6,4]
            nn.ReLU(),
            nn.Linear(6, 6),   # matches [6,6]
            nn.ReLU(),
            nn.Linear(6, 1)    # matches [1,6]
        )

    def forward(self, x):
        return self.model(x)

In [55]:
model = ANNModel()
state_dict = torch.load("best_model.pt")

print("Model keys:", model.state_dict().keys())
print("Saved keys:", state_dict.keys())

Model keys: odict_keys(['model.0.weight', 'model.0.bias', 'model.2.weight', 'model.2.bias', 'model.4.weight', 'model.4.bias'])
Saved keys: odict_keys(['model.0.weight', 'model.0.bias', 'model.2.weight', 'model.2.bias', 'model.4.weight', 'model.4.bias'])


In [47]:
predicted_df = pd.DataFrame(test_preds.numpy(), columns=["Predicted Values"])
actual_df = pd.DataFrame(y_test.values, columns=["Actual Values"])

pd.concat([predicted_df, actual_df], axis=1)

,Predicted Values,Actual Values
0,435.809845,433.27
1,437.769104,438.16
2,461.984680,458.42
3,477.107819,480.82
4,435.892456,441.41
...,...,...
1909,452.115387,456.70
1910,432.328125,438.04
1911,468.218842,467.80
1912,431.802826,437.14


In [52]:
print(type(torch.load("best_model.pt")))

<class 'collections.OrderedDict'>


In [53]:
model = ANNModel()
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

RuntimeError: Error(s) in loading state_dict for ANNModel:
	size mismatch for model.0.weight: copying a param with shape torch.Size([6, 4]) from checkpoint, the shape in current model is torch.Size([16, 4]).
	size mismatch for model.0.bias: copying a param with shape torch.Size([6]) from checkpoint, the shape in current model is torch.Size([16]).
	size mismatch for model.2.weight: copying a param with shape torch.Size([6, 6]) from checkpoint, the shape in current model is torch.Size([8, 16]).
	size mismatch for model.2.bias: copying a param with shape torch.Size([6]) from checkpoint, the shape in current model is torch.Size([8]).
	size mismatch for model.4.weight: copying a param with shape torch.Size([1, 6]) from checkpoint, the shape in current model is torch.Size([1, 8]).

In [54]:
state_dict = torch.load("best_model.pt")

for key in state_dict:
    print(key, state_dict[key].shape)

model.0.weight torch.Size([6, 4])
model.0.bias torch.Size([6])
model.2.weight torch.Size([6, 6])
model.2.bias torch.Size([6])
model.4.weight torch.Size([1, 6])
model.4.bias torch.Size([1])


In [56]:
torch.save(model, "full_model.pt")